# Track B (Behavioural Economics): starter notebook - simulated risky choice

Predict the probability that `chose_risky` = 1. Primary metric: **ROC-AUC**. The test set contains **participants never seen in training**, so validation should hold out whole participants.

*IML Midterm Project, CS-3410 / MAT-3211-1, Ashoka University. Data are simulated.*

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

SEED = 0
SLUG = "iml2026-track-b-simulated-risky-choice"
# Works on Kaggle (dataset attached), locally (./data/<folder>) or via IML_DATA_DIR.
candidates = [os.environ.get("IML_DATA_DIR", ""), f"/kaggle/input/{SLUG}", "data/track_b_behavioural_risky_choice", "data"]
DATA = next(Path(c) for c in candidates if c and (Path(c) / "train.csv").exists())
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
print(DATA, train.shape, test.shape)
train.head()

## 1. Remove what you will not have at prediction time
`realised_payout_inr` is paid *after* the choice and is absent from `test.csv`; it almost reveals the answer. Drop it.

In [ ]:
ID, TARGET, GROUP = "decision_id", "chose_risky", "participant_id"
train = train.drop(columns=["realised_payout_inr"])
print(train[TARGET].mean().round(3), train[GROUP].nunique(), "participants")

## 2. A validation split that matches the test set
`GroupKFold` keeps every participant's 40 decisions together, so the model is always validated on people it has not seen.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
cv = GroupKFold(n_splits=5)
groups = train[GROUP]
y = train[TARGET]

def evaluate(model, X):
    p = cross_val_predict(model, X, y, cv=cv, groups=groups, method="predict_proba")[:, 1]
    return round(roc_auc_score(y, p), 3)

## 3. Trivial baseline and a transparent rule
A constant prediction has AUC 0.5. The expected-value rule (choose the option with the higher expected payoff) is a useful benchmark that needs no training.

In [ ]:
FEATURES = ["prob_high", "sure_amount_inr"]
print("constant:", evaluate(DummyClassifier(strategy="prior"), train[FEATURES]), "(not exactly 0.5: each fold predicts its own constant)")
ev_gap = train.prob_high * train.risky_high_inr + (1 - train.prob_high) * train.risky_low_inr - train.sure_amount_inr
print("expected-value rule:", round(roc_auc_score(y, ev_gap), 3))

## 4. A deliberately small first model
Two attributes only (the probability and the sure amount): no outcomes, no frame, no context, no participant information.

In [ ]:
model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000))
print("small logistic:", evaluate(model, train[FEATURES]))

## 5. Fit on all training data and write the submission

In [ ]:
model.fit(train[FEATURES], y)
submission = pd.DataFrame({ID: test[ID], TARGET: model.predict_proba(test[FEATURES])[:, 1]})

In [ ]:
# Check the file against the required format before uploading it.
assert list(submission.columns) == list(sample.columns), "columns must match sample_submission.csv"
assert len(submission) == len(sample) and set(submission[ID]) == set(sample[ID]), "every test id exactly once"
assert submission[TARGET].notna().all()
submission.to_csv("submission.csv", index=False)
print("wrote submission.csv", submission.shape)
submission.head()

## Where to go from here
This notebook gives you a sound *protocol*, not a good model. Your project adds a research question, at least three baselines, at least two controlled experiments, error analysis and limitations (see the project guide). Keep every new idea inside the same validation scheme. The data are **simulated**; say so in your report.